# Credit Manager - Database Initialization Test
Este notebook inicializa la base de datos SQLite y verifica que todas las tablas mapeadas en SQLAlchemy se hayan creado correctamente a partir de nuestro módulo `src/database`.

In [ ]:
import sys
import os

from importlib import reload

# Asegurar que el path apunte a la raíz para encontrcar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
"""
Notebook Cell: Database Reset & Modular Seeding
Description: Drops tables, recreates them, and calls the external seeding module.
Author: Juan Martín Carini
Date: 2026-05-11
"""


import src.database  # noqa: E402
import src.database.seed_geography  # noqa: E402

def reset_and_seed():
    print("Iniciando reset de base de datos...")
    
    # 1. Limpieza total
    src.database.Base.metadata.drop_all(bind=src.database.engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Reconstrucción del esquema
    src.database.Base.metadata.create_all(bind=src.database.engine)
    print("✅ Estructura de tablas recreada correctamente.")

    # 3. Carga de datos maestros usando el nuevo módulo
    db = src.database.SessionLocal()
    try:
        print("Poblando tablas geográficas...")
        src.database.seed_geography.seed_provincias(db)
    except Exception as e:
        db.rollback()
        print(f"❌ Error durante el seeding: {e}")
    finally:
        db.close()

if __name__ == "__main__":
    reset_and_seed()

In [ ]:
import src.logic.amortization as amortization
reload(amortization)
import src.logic.origination as origination
reload(origination)
from src.database.connection import SessionLocal
from src.database.models import Cliente, SexoEnum, SocioComercial
import datetime

reload(origination)

db = SessionLocal()

try:
    originator = origination.LoanOriginator(db_session=db)
    
    # Armamos el diccionario con los datos del nuevo cliente
    datos_cliente = {
        "cuil": "27334445558",
        "documento": "33444555",
        "apellido": "Gómez",
        "nombre": "Ana",
        "sexo": SexoEnum.FEMENINO,
        "remuneracion": 850000.0,
        "mail": "ana.gomez@ejemplo.com"
    }

    # Disparamos la creación dual
    nuevo_credito = originator.originate_with_new_client(
        client_data=datos_cliente,
        capital= 1012145.75,
        tna_c_iva=1.71,
        term=12,
        partner_id=1,  # Mutual Interna 13 (u otro ID válido)
        issuance_date=datetime.date.today(),
        due_day=15
    )
    
    print(f"✅ Cliente {datos_cliente['apellido']} y Crédito {nuevo_credito.id} registrados con éxito.")
    
except Exception as e:
    print(f"❌ Error: {e}")
finally:
    db.close()

In [ ]:
from src.reports import saldos

df = saldos()
df.loc[("Total",""), ["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].sum()

print(f"Total: $ {df.loc["Total", ["Capital", "Interés", "IVA"]].sum().sum():,.2f}")

df[["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].map("$ {:,.2f}".format)

df